# BENCHMARK TÁI HIỆN BẢNG 3: VAR + MASC vs VAR BASELINE
**Visual AutoRegressive (VAR-d16, ~310M params) tích hợp Manifold-Aligned Semantic Clustering (MASC k=2,048)**
*Đánh giá trên Kaggle 2x NVIDIA T4 (16 GB GDDR6) — Chuẩn 10-Class ImageNet-1K (10,000 samples)*

---

### Mục tiêu nghiên cứu (Theo Table 3 trong Paper MASC gốc):
1. **Chứng minh tính Universal Plugin**: MASC không chỉ hỗ trợ mô hình Next-Token (LlamaGen) mà còn nâng cấp trực tiếp mô hình Next-Scale (VAR, NeurIPS 2024).
2. **So sánh sòng phẳng**:
   - **`VAR-d16 Baseline (V=4,096)`**: FID = **30.34**, IS = **8.74** *(Đã đo đạc chuẩn ở notebook var.ipynb)*.
   - **`VAR-d16 + MASC (k=2,048)`**: Thu hẹp vocab từ 4,096 xuống 2,048 cụm đa tạp, đo độ cải thiện FID, IS và tốc độ hội tụ.
   - **`LlamaGen-L + MASC + UPipe`**: Trường phái Next-Token với MASC (FID = **33.45**).


In [ ]:
# ============================================================
# CELL 1 — Thiết lập môi trường & Clone Repositories
# ============================================================
import subprocess, os, sys, shutil, torch

WORK_DIR      = '/kaggle/working'
VAR_DIR       = f'{WORK_DIR}/VAR'
BENCHMARK_DIR = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR   = f'{WORK_DIR}/weights'
CODES_DIR     = f'{WORK_DIR}/codes'
MASC_DIR      = f'{WORK_DIR}/masc-upgrade-ann'
LLAMA_DIR     = f'{WORK_DIR}/LlamaGen'
CKPT_VAR_MASC_DIR = f'{WORK_DIR}/checkpoints_var_masc'

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(CKPT_VAR_MASC_DIR, exist_ok=True)

# 1. Cài đặt các gói phụ thuộc
print('Cài đặt dependencies...')
subprocess.run('pip install -q huggingface_hub pyyaml pandas tabulate scipy', shell=True)

# 2. Clone / Cập nhật Repositories
def clone_repo(url, target):
    if not os.path.exists(target):
        print(f'>>> git clone {url} {target}')
        subprocess.run(f'git clone {url} {target}', shell=True, check=True)
    else:
        print(f'>>> {target} đã có sẵn, cập nhật git pull...')
        subprocess.run(f'git -C {target} pull', shell=True)

clone_repo('https://github.com/nta2112/masc-upgrade-ann', MASC_DIR)
clone_repo('https://github.com/FoundationVision/LlamaGen', LLAMA_DIR)
clone_repo('https://github.com/nta2112/Masce-baseline-benchmark', BENCHMARK_DIR)
clone_repo('https://github.com/FoundationVision/VAR', VAR_DIR)

# 3. Kiểm tra thông số GPU
print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
print(f'Số lượng GPU    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'   GPU {i} : {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)')
print('=' * 60)


In [ ]:
# ============================================================
# CELL 2 — Tải Weights Pretrained của VAR & LlamaGen
# Multi-Scale VQ-VAE + VAR-d16 (~310M parameters)
# ============================================================
from huggingface_hub import hf_hub_download
import os

VAR_WEIGHTS = {
    'vae': ('vae_ch160v4096z32.pth', 'FoundationVision/var'),
    'var': ('var_d16.pth', 'FoundationVision/var'),
}

for name, (fname, repo) in VAR_WEIGHTS.items():
    dest = f'{WEIGHTS_DIR}/{fname}'
    if not os.path.exists(dest):
        print(f'Đang tải {fname} từ {repo}...')
        hf_hub_download(repo_id=repo, filename=fname, local_dir=WEIGHTS_DIR)
        print(f'✅ Đã tải: {fname}')
    else:
        print(f'✅ Đã có sẵn: {fname} ({os.path.getsize(dest)/1e6:.1f} MB)')

# Tải VQ-VAE của LlamaGen từ FoundationVision/LlamaGen (Public repo, không cần HF_TOKEN)
VQ_NAME = 'vq_ds16_c2i.pt'
VQ_DEST = f'{WEIGHTS_DIR}/{VQ_NAME}'
if not os.path.exists(VQ_DEST):
    print(f'Đang tải {VQ_NAME} từ FoundationVision/LlamaGen...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen', filename=VQ_NAME, local_dir=WEIGHTS_DIR)
    print(f'✅ Đã tải: {VQ_NAME} ({os.path.getsize(VQ_DEST)/1e6:.1f} MB)')
else:
    print(f'✅ Đã có sẵn: {VQ_NAME} ({os.path.getsize(VQ_DEST)/1e6:.1f} MB)')


In [ ]:
# ============================================================
# CELL 3 — Chuẩn bị Dữ liệu Huấn luyện & Reference Batch 10-Class
# 10 Class x 1,000 ảnh = 10,000 ảnh chuẩn
# ============================================================
import os, glob, io, tarfile, time, shutil, subprocess
import numpy as np, torch
from pathlib import Path
from huggingface_hub import snapshot_download

existing_npy = sorted(Path(CODES_DIR).glob('shard_*.npy'))
labels_path  = f'{CODES_DIR}/labels.npy'
REF_NPZ      = f'{WORK_DIR}/ref_10class_balanced_1000.npz'

# Xóa cache file cũ nếu có để tránh dùng lại file chỉ chứa 1 class cá
old_ref = f'{WORK_DIR}/ref_10class_1000.npz'
if os.path.exists(old_ref):
    try: os.remove(old_ref)
    except Exception: pass

# 1. Trích xuất 10,000 samples token codes nếu chưa có
if not (existing_npy and os.path.exists(labels_path) and len(np.load(labels_path)) == 10000):
    WDS_DIR = f'{WORK_DIR}/wds_cache'
    os.makedirs(WDS_DIR, exist_ok=True)
    print('Đang tải imagenet-llamagen-cache để lấy đúng 10,000 ảnh huấn luyện...')
    snapshot_download(
        repo_id='Efficient-Large-Model/imagenet-llamagen-cache',
        repo_type='dataset', local_dir=WDS_DIR,
        ignore_patterns=['*.json', '*.yaml', 'README*'],
    )
    tar_files = sorted(Path(WDS_DIR).glob('**/*.tar'))
    tar_path = tar_files[0]
    print(f'Đang trích xuất 10 class từ {tar_path.name}...')

    import urllib.request, json
    u = 'https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json'
    d = json.loads(urllib.request.urlopen(u).read().decode())
    SYNSET_TO_ID = {d[str(i)][0]: i for i in range(1000)}

    TARGET_CLASSES = list(range(10))
    MAX_PER_CLASS  = 1000
    class_counts   = {c: 0 for c in TARGET_CLASSES}
    shard_codes, all_labels = [], []
    shard_idx, total_imgs = 0, 0
    SHARD_SIZE = 512

    with tarfile.open(tar_path, 'r:*') as tf:
        for member in tf:
            if not member.name.endswith('.npz'): continue
            parts = member.name.split('/')
            if len(parts) < 2: continue
            label = SYNSET_TO_ID.get(parts[-2], -1)
            if label not in class_counts or class_counts[label] >= MAX_PER_CLASS: continue

            f = tf.extractfile(member)
            if f is None: continue
            try:
                npz = np.load(io.BytesIO(f.read()))
                codes = npz['indices'].flatten().astype(np.int32)
            except Exception: continue
            if len(codes) != 256: continue

            shard_codes.append(codes)
            all_labels.append(label)
            class_counts[label] += 1
            total_imgs += 1

            if len(shard_codes) >= SHARD_SIZE:
                np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
                shard_codes = []; shard_idx += 1
            if total_imgs >= 10000: break

    if shard_codes:
        np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
    np.save(labels_path, np.array(all_labels, dtype=np.int32))
    shutil.rmtree(WDS_DIR, ignore_errors=True)
    print(f'✅ Đã chuẩn bị 10,000 samples LlamaGen codes trong {CODES_DIR}!')
else:
    print('✅ Đã có sẵn 10,000 samples LlamaGen codes trên đĩa!')

# 2. Tạo Reference Batch (1,000 ảnh thật cân bằng 10 class) để tính FID chuẩn
if not os.path.exists(REF_NPZ):
    print('Đang tạo Reference Batch 10-Class CÂN BẰNG (100 ảnh/class x 10 class = 1,000 ảnh)...')
    LLAMA_REPO = f'{WORK_DIR}/LlamaGen'
    if not os.path.exists(LLAMA_REPO):
        subprocess.run(f'git clone https://github.com/FoundationVision/LlamaGen {LLAMA_REPO}', shell=True)
    if LLAMA_REPO not in sys.path: sys.path.insert(0, LLAMA_REPO)

    from tokenizer.tokenizer_image.vq_model import VQ_models
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    vq = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8)
    vq_ckpt = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
    st = torch.load(vq_ckpt, map_location='cpu')
    vq.load_state_dict(st.get('model', st))
    vq.to(device).eval()

    shards = sorted(Path(CODES_DIR).glob('shard_*.npy'))
    all_codes = np.concatenate([np.load(f) for f in shards], axis=0)

    if os.path.exists(labels_path):
        all_labels = np.load(labels_path)
        ref_indices = []
        for c in range(10):
            c_idxs = np.where(all_labels == c)[0][:100]
            ref_indices.extend(c_idxs)
        ref_indices = np.array(ref_indices)
        ref_codes = all_codes[ref_indices]
        print(f'   -> Đã trích xuất chính xác 100 ảnh/class cho 10 class (Tổng: {len(ref_codes)} ảnh)')
    else:
        rng = np.random.default_rng(42)
        ref_indices = rng.choice(len(all_codes), size=1000, replace=False)
        ref_codes = all_codes[ref_indices]
        print(f'   -> Đã lấy ngẫu nhiên 1,000 ảnh trên toàn bộ shards')

    ref_imgs = []
    bs = 20
    with torch.no_grad():
        for i in range(0, len(ref_codes), bs):
            b_c = torch.tensor(ref_codes[i:i+bs], device=device, dtype=torch.long)
            imgs = vq.decode_code(b_c, [len(b_c), 8, 16, 16])
            imgs = ((imgs.clamp(-1, 1) + 1.0) / 2.0 * 255.0).permute(0, 2, 3, 1).to(torch.uint8).cpu().numpy()
            ref_imgs.append(imgs)

    ref_arr = np.concatenate(ref_imgs, axis=0)
    np.savez_compressed(REF_NPZ, arr_0=ref_arr)
    print(f'✅ Reference Batch hoàn thành: {REF_NPZ} ({ref_arr.shape})')
    del vq; torch.cuda.empty_cache()
else:
    print(f'✅ Reference Batch đã có sẵn: {REF_NPZ}')



In [ ]:
# ============================================================
# CELL 4 — XÂY DỰNG MASC COARSE MAPPING CHO VAR (V=4,096 -> k=2,048)
# Thuật toán MASC Algorithm 1 (Đồ thị đa tạp & Gom cụm dưới-lên)
# ============================================================
import os, sys, subprocess

MASC_BUILD_SCRIPT = f'{BENCHMARK_DIR}/baselines/03_var/build_var_masc_mapping.py'
VAR_MASC_MAP_NPZ  = f'{WORK_DIR}/var_masc_mapping_k2048.npz'
VAR_MASC_MAP_PT   = f'{WORK_DIR}/var_masc_mapping_k2048.pt'

if os.path.exists(VAR_MASC_MAP_PT):
    print(f'✅ MASC Mapping cho VAR đã tồn tại sẵn tại:
   {VAR_MASC_MAP_PT}')
else:
    print('Khởi chạy xây dựng cây gom cụm MASC cho Codebook VAR V=4096 -> k=2048...')
    cmd = [
        'python', MASC_BUILD_SCRIPT,
        '--vae-ckpt',  f'{WEIGHTS_DIR}/vae_ch160v4096z32.pth',
        '--k',         '2048',
        '--out',       VAR_MASC_MAP_NPZ,
        '--masc-repo', MASC_DIR,
    ]
    proc = subprocess.run(cmd, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError('Lỗi khi xây dựng MASC mapping cho VAR!')
    print('✅ Xây dựng MASC Coarse Mapping k=2048 thành công!')


In [ ]:
# ============================================================
# CELL 5 — Trích xuất VAR Multi-Scale Tokens (680 tokens/sample)
# Phục vụ cho huấn luyện VAR (Tự động bỏ qua nếu đã có sẵn)
# ============================================================
import os, sys, time, gc, glob, torch, numpy as np
from pathlib import Path
from tqdm import tqdm

VAR_TOKENS_PATH = f'{WORK_DIR}/var_tokens_10class.pt'
VAR_LABELS_PATH = f'{WORK_DIR}/var_labels_10class.pt'

if os.path.exists(VAR_TOKENS_PATH) and os.path.exists(VAR_LABELS_PATH):
    print(f'✅ Đã có sẵn VAR tokens cache: {VAR_TOKENS_PATH} ({os.path.getsize(VAR_TOKENS_PATH)/1e6:.1f} MB)')
else:
    print('Bắt đầu chuyển đổi 10,000 ảnh sang VAR Multi-Scale Tokens...')
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    if VAR_DIR not in sys.path: sys.path.insert(0, VAR_DIR)
    from models.vqvae import VQVAE

    patch_nums = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)
    var_vae = VQVAE(
        vocab_size=4096, z_channels=32, ch=160, share_quant_resi=4,
        v_patch_nums=patch_nums, test_mode=True,
    ).to(device)
    vae_state = torch.load(f'{WEIGHTS_DIR}/vae_ch160v4096z32.pth', map_location='cpu')
    var_vae.load_state_dict(vae_state.get('model', vae_state), strict=True)
    var_vae.eval()

    if LLAMA_DIR not in sys.path: sys.path.insert(0, LLAMA_DIR)
    from tokenizer.tokenizer_image.vq_model import VQ_models

    lg_vae = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8)
    lg_st = torch.load(f'{WEIGHTS_DIR}/vq_ds16_c2i.pt', map_location='cpu')
    lg_vae.load_state_dict(lg_st.get('model', lg_st))
    lg_vae.to(device).eval()

    shards = sorted(Path(CODES_DIR).glob('shard_*.npy'))
    all_codes = np.concatenate([np.load(f) for f in shards], axis=0)
    all_labels = np.load(f'{CODES_DIR}/labels.npy')

    bs = 20
    all_var_tokens = []
    t0 = time.time()

    with torch.inference_mode():
        for i in tqdm(range(0, len(all_codes), bs), desc='Extracting VAR Tokens'):
            b_c = torch.tensor(all_codes[i:i+bs], device=device, dtype=torch.long)
            cur_bs = len(b_c)
            imgs = lg_vae.decode_code(b_c, [cur_bs, 8, 16, 16]).clamp(-1, 1)
            gt_idx_Bl = var_vae.img_to_idxBl(imgs)
            gt_BL = torch.cat(gt_idx_Bl, dim=1).cpu()
            all_var_tokens.append(gt_BL)

    all_var_tokens = torch.cat(all_var_tokens, dim=0)
    torch.save(all_var_tokens, VAR_TOKENS_PATH)
    torch.save(torch.from_numpy(all_labels).long(), VAR_LABELS_PATH)

    print(f'✅ Trích xuất xong trong {time.time()-t0:.1f}s!')
    print(f'VAR Tokens Shape: {all_var_tokens.shape} | Dung lượng: {os.path.getsize(VAR_TOKENS_PATH)/1e6:.1f} MB')

    del var_vae, lg_vae
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# CELL 6 — HUẤN LUYỆN 20 EPOCHS CHO VAR + MASC (k=2,048)
# Khởi chạy song song trên 2x GPU T4 bằng torchrun DDP
# ============================================================
import subprocess, os, glob, torch

VAR_MASC_FINAL_CKPT = f'{CKPT_VAR_MASC_DIR}/ckpt_var_masc_10class_final.pt'

if os.path.exists(VAR_MASC_FINAL_CKPT):
    print('=' * 70)
    print(f'✅ PHÁT HIỆN CHECKPOINT VAR + MASC 10-CLASS CÓ SẴN:')
    print(f'   -> {VAR_MASC_FINAL_CKPT}')
    print(f'   -> Dung lượng: {os.path.getsize(VAR_MASC_FINAL_CKPT)/1e6:.1f} MB')
    print('=' * 70)
    print('⏩ BỎ QUA HUẤN LUYỆN, CHUYỂN THẲNG SANG CELL 7 ĐỂ SINH ẢNH & ĐÁNH GIÁ!')
else:
    print('Khởi động huấn luyện 20 Epochs cho VAR-d16 + MASC (k=2048) trên 2x GPU T4...')
    TRAIN_SCRIPT = f'{BENCHMARK_DIR}/baselines/03_var/train_var_masc.py'

    env = os.environ.copy()
    env['PYTHONPATH'] = f'{VAR_DIR}:{MASC_DIR}:{BENCHMARK_DIR}:' + env.get('PYTHONPATH', '')
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    env['OMP_NUM_THREADS'] = '2'

    cmd = [
        'torchrun', '--nproc_per_node=2', '--master_port=29515',
        TRAIN_SCRIPT,
        '--var-repo',     VAR_DIR,
        '--vae-ckpt',     f'{WEIGHTS_DIR}/vae_ch160v4096z32.pth',
        '--var-ckpt',     f'{WEIGHTS_DIR}/var_d16.pth',
        '--mapping-path', VAR_MASC_MAP_PT,
        '--tokens-path',  f'{WORK_DIR}/var_tokens_10class.pt',
        '--labels-path',  f'{WORK_DIR}/var_labels_10class.pt',
        '--depth',        '16',
        '--k',            '2048',
        '--epochs',       '20.0',    # Đúng 20 epochs tương đương VAR baseline!
        '--batch-size',   '8',       # 8 per GPU x 2 accum x 2 GPUs = Global batch 32
        '--grad-accum',   '2',
        '--lr',           '1e-4',
        '--min-lr',       '1e-6',
        '--weight-decay', '0.05',
        '--out-dir',      CKPT_VAR_MASC_DIR,
        '--save-every',   '5',
        '--log-every',    '50'
    ]

    print('
Lệnh khởi chạy huấn luyện song song 2 GPU:')
    print(' '.join(cmd))
    print('=' * 70)

    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)

    proc.wait()
    print(f'\nExit code: {proc.returncode}')
    if proc.returncode == 0:
        print('✅ HUẤN LUYỆN 20 EPOCHS CHO VAR + MASC HOÀN TẤT THÀNH CÔNG!')
    else:
        print('❌ Lỗi trong quá trình huấn luyện, xem log bên trên!')


In [ ]:
# ============================================================
# CELL 7 — Sinh 1,000 Ảnh Mẫu từ VAR + MASC (Song song 2x T4)
# Tái lập cơ chế MASC Sampling: coarse logits -> fine decode -> VQ-VAE
# ============================================================
import subprocess, os, sys

INFER_SCRIPT = f'{BENCHMARK_DIR}/baselines/03_var/var_masc_infer.py'
SAMPLES_OUT  = f'{WORK_DIR}/samples_var_masc_finetuned_1000.npz'

cmd = [
    'python', INFER_SCRIPT,
    '--var-repo',     VAR_DIR,
    '--vae-ckpt',     f'{WEIGHTS_DIR}/vae_ch160v4096z32.pth',
    '--ckpt',         f'{CKPT_VAR_MASC_DIR}/ckpt_var_masc_10class_final.pt',
    '--mapping-path', VAR_MASC_MAP_PT,
    '--depth',        '16',
    '--k',            '2048',
    '--num-samples',  '1000',
    '--batch-size',   '16',
    '--cfg',          '4.0',
    '--top-k',        '900',
    '--top-p',        '0.96',
    '--out',          SAMPLES_OUT
]

print('Bắt đầu sinh 1,000 ảnh song song trên cả 2 GPU T4...')
print(' '.join(cmd))
print('=' * 70)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ Sinh 1,000 ảnh thành công: {SAMPLES_OUT}')
else:
    print('\n❌ Lỗi trong quá trình sinh ảnh!')


In [ ]:
# ============================================================
# CELL 8 — ĐÁNH GIÁ METRICS: FID, IS, Precision, Recall
# So sánh sòng phẳng trên cùng Reference Batch ref_10class_balanced_1000.npz
# ============================================================
import subprocess, os, sys

SAMPLES_OUT = f'{WORK_DIR}/samples_var_masc_finetuned_1000.npz'
REF_NPZ     = f'{WORK_DIR}/ref_10class_balanced_1000.npz'

eval_cmd = f'CUDA_VISIBLE_DEVICES=1 python {LLAMA_DIR}/evaluations/c2i/evaluator.py {REF_NPZ} {SAMPLES_OUT}'
print('Khởi chạy LlamaGen Evaluator trên GPU 1 (dành trọn VRAM cho TF Inception-v3):')
print(eval_cmd)
print('=' * 70)

proc = subprocess.Popen(eval_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n✅ HOÀN TẤT ĐÁNH GIÁ VAR + MASC!')


In [ ]:
# ============================================================
# CELL 9 — BẢNG ĐỐI SÁNH TOÀN DIỆN (TABLE 1 & TABLE 3 REPRODUCTION)
# Dữ liệu thực tế đo đạc trực tiếp trên cùng tập 10-Class ImageNet (20 Epochs)
# ============================================================
import pandas as pd

table_benchmark = [
    {
        "Phương pháp": "Vanilla LlamaGen-L (Baseline Flat)",
        "Trường phái": "Next-Token (N=16,384)",
        "Fine-tuning": "Zero-shot Pretrained",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "45.47",
        "IS ↑": "8.11",
        "Precision ↑": "0.744",
        "Recall ↑": "0.149"
    },
    {
        "Phương pháp": "k-Means Clustering Baseline",
        "Trường phái": "Next-Token (k=4,096)",
        "Fine-tuning": "20 Epochs",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "31.42",
        "IS ↑": "8.85",
        "Precision ↑": "0.705",
        "Recall ↑": "0.533"
    },
    {
        "Phương pháp": "LlamaGen-L + MASC + UPipe (Ours)",
        "Trường phái": "Next-Token (k=4,096)",
        "Fine-tuning": "20 Epochs",
        "Tokens/Ảnh": "256",
        "10-Class FID ↓": "33.45",
        "IS ↑": "8.75",
        "Precision ↑": "0.747",
        "Recall ↑": "0.499"
    },
    {
        "Phương pháp": "VAR-d16 Baseline (Tian et al. 2024)",
        "Trường phái": "Next-Scale (V=4,096)",
        "Fine-tuning": "20 Epochs",
        "Tokens/Ảnh": "680 (10 scales)",
        "10-Class FID ↓": "30.34",
        "IS ↑": "8.74",
        "Precision ↑": "0.684",
        "Recall ↑": "0.653"
    },
    {
        "Phương pháp": "VAR-d16 + MASC (Table 3 Reproduction)",
        "Trường phái": "Next-Scale (k=2,048)",
        "Fine-tuning": "20 Epochs",
        "Tokens/Ảnh": "680 (10 scales)",
        "10-Class FID ↓": "[Điền từ Cell 8]",
        "IS ↑": "[Điền từ Cell 8]",
        "Precision ↑": "[Điền từ Cell 8]",
        "Recall ↑": "[Điền từ Cell 8]"
    }
]

df = pd.DataFrame(table_benchmark)
print(df.to_string(index=False))

print("\n" + "="*85)
print("BẢNG MARKDOWN DÙNG CHO BÀI BÁO (LATEX / MARKDOWN REPORT):")
print("="*85)
print(df.to_markdown(index=False))
